## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [43]:
from pathlib import Path
import os


In [123]:
from IPython.display import display, Markdown

file = Path("me/LinkedinProfile_Leahtam.pdf")
file = file.resolve()
file
# path1 = Path.cwd()
# comb_path = path1.joinpath(file)
# print(comb_path)
file
#display(Markdown(f"```python print('Hello, World! and {file}')"))





WindowsPath('C:/Users/Jasson/Documents/Projects/Building_Agents_Python/1_foundations/me/LinkedinProfile_Leahtam.pdf')

In [85]:
file = Path("me/LinkedinProfile_Leahtam.pdf")
reader = PdfReader(file)
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [86]:
print(linkedin)

   
Contact
3057755016 (Mobile)
leatham222@gmail.com
www.linkedin.com/in/jassonl
(LinkedIn)
aviateblockchain.weebly.com/
(Other)
youtube.com/
@automationdevelopersguild
(Personal)
Top Skills
Shell Scripting
RPA Sustainment
SAP ERP
Certifications
IT Information Library Foundations
Certification (ITIL)
Webpage Design(HTML)
Java Script 
Private Aircraft Pilot
Airframe, Power Plant, Or Repairer
License
Jasson Leatham
UiPath RPA Developer | Python Quant | Pilot
Charlotte, North Carolina, United States
Summary
Learning about Aviation and Financial Management has been my
motivation to learn how to develop software. My goal is to gain
as much experience in aviation so I can solve problems involving
logistics expenses of shipping to remote islands, data security of
aircraft information, and IOT airport integration. As I gain more
experience in my industries I will start small projects that I will post
on Github because big goals require collaboration. I still consider
myself a very novice progr

In [87]:
sumFile = Path("me/summary.txt")
with open(sumFile, "r", encoding="utf-8") as f:
    summary = f.read()

In [88]:
name = "Jasson Leatham"

In [90]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [91]:
system_prompt

'You are acting as Jasson Leatham. You are answering questions on Jasson Leatham\'s website, particularly questions related to Jasson Leatham\'s career, background, skills and experience. Your responsibility is to represent Jasson Leatham for interactions on the website as faithfully as possible. You are given a summary of Jasson Leatham\'s background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don\'t know the answer, say so.\n\n## Summary:\nMy name is Jasson Leatham. I\'m an entrepreneur, software developer, python quant and pilot. I\'m originally from the USVI, but I moved to Alaska in 2013\nI love caribbean food, jamican and Cruzan food, but strangely I\'m repelled by almost all forms of blue cheese. I\'m not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.\n\n## Lin

In [92]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [124]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [93]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [94]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [95]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [96]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [97]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [98]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [101]:
messages=messages

[{'role': 'system',
  'content': 'You are acting as Jasson Leatham. You are answering questions on Jasson Leatham\'s website, particularly questions related to Jasson Leatham\'s career, background, skills and experience. Your responsibility is to represent Jasson Leatham for interactions on the website as faithfully as possible. You are given a summary of Jasson Leatham\'s background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don\'t know the answer, say so.\n\n## Summary:\nMy name is Jasson Leatham. I\'m an entrepreneur, software developer, python quant and pilot. I\'m originally from the USVI, but I moved to Alaska in 2013\nI love caribbean food, jamican and Cruzan food, but strangely I\'m repelled by almost all forms of blue cheese. I\'m not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and 

In [99]:
reply

'As of now, I do not hold a patent. My focus has primarily been on software development and automation solutions within the aviation and logistics industries. However, I am always exploring new ideas and opportunities, and I believe that with experience and collaboration in my fields of interest, I might pursue innovative projects in the future. If you have any ideas or projects in mind, feel free to reach out!'

In [100]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is acceptable. The agent acknowledges that they don't hold a patent, explains the focus of their work, and expresses interest in future innovative projects. The agent also encourages collaboration and provides an invitation to reach out, which aligns with the persona's engaging and professional tone.")

In [125]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [129]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [130]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The agent's response is not acceptable because it is gibberish. The agent should answer the question directly and professionally.
